# From a raw recording to clean epochs

This notebook walks one EEG recording through the library, in the
recommended order: check quality, find bad channels, run a saved
preprocessing configuration, check quality again, and look at the result.

It uses a **synthetic** recording so it runs anywhere without downloads,
and so we know the truth: one channel (T7) is broken, there are six large
blinks, the mains frequency is 50 Hz, and "target" trials carry a
P300-like response over parietal sites.
Replace the first cell with your own data, e.g.
`raw = mne.io.read_raw_edf("recording.edf", preload=True)`.

In [ ]:
# The library returns matplotlib Figure objects; "inline" makes Jupyter draw them.
%matplotlib inline
import numpy as np
import mne

from eeg_decoder.io import to_raw

mne.set_log_level("warning")  # keep MNE's filter-design messages out of the output

SFREQ = 250.0
CHANNELS = ["Fp1", "Fp2", "F3", "Fz", "F4", "T7", "C3", "Cz", "C4", "T8",
            "P3", "Pz", "P4", "O1", "Oz", "O2"]


def synthetic_recording(seed=0, seconds=120.0):
    """16-channel EEG in microvolts plus a stim channel (codes 1 and 2)."""
    rng = np.random.default_rng(seed)
    n = int(seconds * SFREQ)
    t = np.arange(n) / SFREQ
    pos = mne.channels.make_standard_montage("colin27_1020").get_positions()["ch_pos"]
    xyz = np.array([pos[c] for c in CHANNELS])
    # Spatially smooth background: neighbouring electrodes are correlated.
    spectrum = np.fft.rfft(rng.normal(size=(4, n)), axis=1)
    freqs = np.fft.rfftfreq(n, 1 / SFREQ)
    spectrum[:, 1:] /= np.sqrt(freqs[1:])  # 1/f ("pink") power, like resting EEG
    spectrum[:, 0] = 0
    latent = np.fft.irfft(spectrum, n, axis=1)
    latent = latent / latent.std(axis=1, keepdims=True) * 8
    mixing = np.column_stack([np.ones(len(CHANNELS)), xyz / np.abs(xyz).max(axis=0)])
    eeg = mixing @ latent + rng.normal(0, 2, (len(CHANNELS), n))
    occipital = np.exp(-np.linalg.norm(xyz - pos["Oz"], axis=1) ** 2 / 0.004)
    eeg += np.outer(occipital, 6 * np.sin(2 * np.pi * 10 * t))  # alpha
    eeg += 5 * np.sin(2 * np.pi * 50 * t)  # mains interference, 50 Hz
    # Events every 1.5 s; one in four is a "target" (code 2).
    onsets = np.arange(int(2 * SFREQ), n - int(2 * SFREQ), int(1.5 * SFREQ))
    codes = np.where(rng.random(onsets.size) < 0.25, 2, 1)
    parietal = np.exp(-np.linalg.norm(xyz - pos["Pz"], axis=1) ** 2 / 0.006)
    p300 = 10 * np.exp(-((np.arange(int(0.8 * SFREQ)) / SFREQ - 0.35) / 0.08) ** 2)
    for onset, code in zip(onsets, codes):
        if code == 2:
            eeg[:, onset:onset + p300.size] += np.outer(parietal, p300)
    frontal = np.exp(-np.linalg.norm(xyz - (pos["Fp1"] + pos["Fp2"]) / 2, axis=1) ** 2 / 0.003)
    for blink in rng.uniform(5, seconds - 5, 6):  # six large blinks
        eeg += np.outer(frontal, 250 * np.exp(-((t - blink) / 0.1) ** 2))
    eeg[CHANNELS.index("T7")] = rng.normal(0, 60, n)  # a broken electrode
    stim = np.zeros(n)
    stim[onsets] = codes
    raw = to_raw(np.vstack([eeg, stim]), SFREQ, CHANNELS + ["STI"],
                 ["eeg"] * len(CHANNELS) + ["stim"], units="uV")
    return raw.set_montage("colin27_1020")


raw = synthetic_recording()
raw

`to_raw` converted the microvolt array to volts (MNE's convention) and
left the stim channel's codes untouched.

## 1. Quality before preprocessing

Metrics are descriptive, never pass/fail. The line frequency is never
assumed: pass 50 or 60 Hz explicitly.

In [ ]:
from eeg_decoder.quality import recording_quality

before = recording_quality(raw, line_freq=50.0)
print(before.summary())

## 2. Find bad channels, then decide

Assess on a high-passed copy, review the evidence, and only then mark
channels bad. Assessment is deliberately a separate, reviewed step, not a
pipeline operation.

In [ ]:
from eeg_decoder.artifacts import assess_channels, mark_bad_channels
from eeg_decoder.preprocessing import bandpass

assessment = assess_channels(bandpass(raw, 1.0, 40.0))
print(assessment.summary())

In [ ]:
raw_marked = mark_bad_channels(raw, assessment.bad)
raw_marked.info["bads"]

## 3. A saved, reproducible preprocessing configuration

Each step is `{"op": <function name>, **its arguments}`. The configuration
is validated before any data are touched, and can be saved as JSON next to
your results.

In [ ]:
from eeg_decoder.preprocessing import EEGPipeline, PreprocessingConfig

config = PreprocessingConfig([
    {"op": "notch", "freqs": [50.0]},
    {"op": "bandpass", "l_freq": 0.5, "h_freq": 30.0},
    {"op": "interpolate_bad_channels"},
    {"op": "rereference", "ref": "average"},
    {"op": "make_epochs", "event_source": "stim", "stim_channel": "STI",
     "event_id": {"standard": 1, "target": 2}, "tmin": -0.2, "tmax": 0.8},
    {"op": "baseline_correct", "baseline": [None, 0.0]},
    {"op": "reject_epochs", "reject": {"eeg": 150e-6}},
])
epochs, record = EEGPipeline(config).run(raw_marked, metadata={"subject": "demo"})
epochs

The record states what ran, with which versions, and what each step
produced. Save it with the configuration (e.g. `json.dump(record, f)`).

In [ ]:
for step in record["steps"]:
    print(f"{step['op']:<25}", {k: v for k, v in step["output"].items()
                                 if k in ("sfreq", "highpass", "lowpass", "n_epochs", "bads")})

## 4. Quality after preprocessing

The same steps up to re-referencing, as a continuous recording, show what
filtering did to the spectrum: the 50 Hz peak is gone and the slow drift
is attenuated.

In [ ]:
from eeg_decoder.quality import plot_psd_comparison

continuous = PreprocessingConfig(config.to_dict()["steps"][:4])
clean_raw, _ = EEGPipeline(continuous).run(raw_marked)
print(recording_quality(clean_raw, line_freq=50.0).summary())
plot_psd_comparison(raw_marked, clean_raw, fmax=70.0, labels=("raw", "preprocessed"))

In [ ]:
from eeg_decoder.quality import epochs_quality, plot_rejection_summary

eq = epochs_quality(epochs)
print(eq.summary())
plot_rejection_summary(eq)

## 5. The result: a P300-like response to targets

Averages per condition at Pz. Targets show the positive deflection around
350 ms that was put into the synthetic data; standards do not.

In [ ]:
from matplotlib.figure import Figure

fig = Figure(figsize=(6, 3), layout="constrained")
ax = fig.add_subplot()
for condition, colour in (("standard", "#2a78d6"), ("target", "#eb6834")):
    evoked = epochs[condition].average(picks="eeg")
    ax.plot(evoked.times, evoked.get_data(picks="Pz")[0] * 1e6, color=colour,
            label=f"{condition} (n={evoked.nave})")
ax.axvline(0, color="0.5", lw=0.8)
ax.set(xlabel="time (s)", ylabel="Pz (µV)")
ax.legend(frameon=False)
fig

## 6. Optional: fixed features

Feature extraction is separate from preprocessing and never required.
Each value uses only one epoch's samples, so it cannot leak information
between epochs. Window lengths and bands are always explicit.

In [ ]:
from eeg_decoder.features import band_power, hjorth

bands = {"theta": (4.0, 8.0), "alpha": (8.0, 13.0), "beta": (13.0, 30.0)}
bp = band_power(epochs, bands, window_s=1.0)
X, names = bp.flatten()
print(bp.dims, bp.data.shape, bp.units)
print(X.shape, names[:4])
y = bp.events[:, 2]  # rows align with epochs.events

In [ ]:
h = hjorth(epochs)  # low-pass first: Hjorth parameters are noise-sensitive
print(h.dims, h.coords["parameter"])